### Installation

In [3]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [4]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 65.4 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.12 environment at: /usr
Resolved 18 packages in 57ms                                         
Prepared 1 package in 1.89s                                              
Uninstalled 1 package in 270ms
Installed 1 package in 195ms                                
 - transformers==4.57.6
 + transformers==4.56.2
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 2ms                                            
Prepared 1 package in 75ms                                               
Uninstalled 1 package in 6ms
Installed 1 package in 8ms                                  
 - trl==0.24.0
 + trl==0.22.2


In [5]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Math-1.5B",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = True, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-02-20 04:26:21.704191: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771561581.895946      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771561581.951743      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771561582.393717      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771561582.393775      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771561582.393779      55 computation_placer.cc:177] computation placer alr

INFO 02-20 04:27:02 [__init__.py:244] Automatically detected platform cuda.
ERROR 02-20 04:27:05 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 02-20 04:27:15 [vllm_utils.py:723] Unsloth: Patching vLLM v1 graph capture
INFO 02-20 04:27:15 [vllm_utils.py:752] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Standby mode is enabled. However your setting of `gpu_memory_utilization` will 

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 02-20 04:27:40 [config.py:3371] Casting torch.bfloat16 to torch.float16.
INFO 02-20 04:27:40 [config.py:1472] Using max model len 2048
WARNING 02-20 04:27:40 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 02-20 04:27:43 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=4096.
INFO 02-20 04:27:43 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.2) with config: model='unsloth/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='unsloth/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disabl

...

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/148M [00:00<?, ?B/s]

Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Unsloth: Already have LoRA adapters! We shall skip this step.


### GRPO chat template
Since we're using a base model, we should set a chat template. You can make your own chat template as well!
1. DeepSeek uses `<think>` and `</think>`, but this is **not** necessary - you can customize it however you like!
2. A `system_prompt` is recommended to at least guide the model's responses.

In [6]:
reasoning_start = "<think>" # Acts as <think>
reasoning_end   = "</think>"   # Acts as </think>
solution_start  = "<answer>"
solution_end    = "</answer>"

system_prompt = \
f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within {reasoning_start} {reasoning_end} and answer is enclosed within {solution_start} {solution_end} tags, respectively, i.e., {reasoning_start} reasoning process here {reasoning_end} {solution_start} answer here {solution_end}."""
system_prompt

'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.'

We create a simple chat template below. Notice `add_generation_prompt` includes prepending `<start_working_out>` to guide the model to start its reasoning process.

In [7]:
chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

# Replace with our specific template:
chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

Let's see how our chat template behaves on an example:

In [8]:
tokenizer.apply_chat_template([
    {"role" : "user", "content" : "What is 1+1?"},
    {"role" : "assistant", "content" : f"{reasoning_start}I think it's 2.{reasoning_end}{solution_start}2{solution_end}"},
    {"role" : "user", "content" : "What is 2+2?"},
], tokenize = False, add_generation_prompt = True)

"A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.<|endoftext|>What is 1+1?<think>I think it's 2.</think><answer>2</answer><|endoftext|>What is 2+2?<think>"

### Data Prep
<a name="Data"></a>

In [ ]:
from datasets import load_dataset

# 1. Load dataset từ HuggingFace
dataset = load_dataset("gsm8k", "main")
train_dataset = dataset["train"]
test_dataset = dataset['test']

In [14]:
!pip install math_verify

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 8.9 MB/s eta 0:00:00


In [15]:
from math_verify import parse, verify

def extract_math_answer(solution_text):
    parsed_answer = parse(solution_text)
    return str(parsed_answer)

Let's map the dataset! and see the first row:

In [ ]:
dataset = train_dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["question"]},
    ],
    "answer": extract_math_answer(x["answer"]),
})
dataset[0]

Map:   0%|          | 0/1348 [00:00<?, ? examples/s]

{'problem': 'The sequence of integers in the row of squares and in each of the two columns of squares form three distinct arithmetic sequences. What is the value of $N$?\n\n[asy]\nunitsize(0.35inch);\ndraw((0,0)--(7,0)--(7,1)--(0,1)--cycle);\ndraw((1,0)--(1,1));\ndraw((2,0)--(2,1));\ndraw((3,0)--(3,1));\ndraw((4,0)--(4,1));\ndraw((5,0)--(5,1));\ndraw((6,0)--(6,1));\ndraw((6,2)--(7,2)--(7,-4)--(6,-4)--cycle);\ndraw((6,-1)--(7,-1));\ndraw((6,-2)--(7,-2));\ndraw((6,-3)--(7,-3));\ndraw((3,0)--(4,0)--(4,-3)--(3,-3)--cycle);\ndraw((3,-1)--(4,-1));\ndraw((3,-2)--(4,-2));\nlabel("21",(0.5,0.8),S);\nlabel("14",(3.5,-1.2),S);\nlabel("18",(3.5,-2.2),S);\nlabel("$N$",(6.5,1.8),S);\nlabel("-17",(6.5,-3.2),S);\n[/asy]',
 'level': 'Level 2',
 'type': 'Algebra',
 'solution': 'Since $18 - 14 = 4$, the common difference in the first column of squares is 4, so the number above 14 is $14 - 4 = 10$, and the number above 10 is $10 - 4 = 6$.  This is also the fourth number in the row, so the common differenc

We now prepare our main function which will print out the generated responses and the true answer, along with another reward function which converts text to float via `float` and sees if it's the same.

In [ ]:
from math_verify import parse, verify

# Khởi tạo biến đếm global
global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5

def dr_grpo_reward_fn(prompts, completions, answer, **kwargs):
    global PRINTED_TIMES
    global PRINT_EVERY_STEPS

    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    scores = []
    extracted_texts = []

    for response_text, gt_answer in zip(responses, answer):
        if "<answer>" in response_text and "</answer>" in response_text:
            model_answer = response_text.split("<answer>")[-1].split("</answer>")[0].strip()
            extracted_texts.append(model_answer)

            try:
                parsed_model = parse(model_answer)
                parsed_gt = parse(str(gt_answer))

                if verify(parsed_model, parsed_gt):
                    scores.append(1.0)
                else:
                    scores.append(0.0) 
            except Exception:
                scores.append(0.0)
        else:
            extracted_texts.append("Error")
            scores.append(0.0)

    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print("=" * 60)
        print(f"[{PRINTED_TIMES}] QUESTION: {question[:150]}...")
        print("-" * 60)
        print(f"GROUND TRUTH : {answer[0]}")
        print(f"MODEL RES    : {responses[0][:400]}... [Cắt bớt]")
        print("-" * 60)
        print(f"EXTRACTED RES: {extracted_texts[0]}")
        print(f"SCORE GIVEN  : {scores[0]}")
        print("=" * 60)

    PRINTED_TIMES += 1

    return scores

Get the top 90% prompt length so we don't accidentally truncate them!

Ie we'll remove the top 10% long prompts.

In [18]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt = True, tokenize = True)},
    batched = True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

# Filter only samples smaller than 90% max length
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

Map:   0%|          | 0/1348 [00:00<?, ? examples/s]

A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.<|endoftext|>The sequence of integers in the row of squares and in each of the two columns of squares form three distinct arithmetic sequences. What is the value of $N$?

[asy]
unitsize(0.35inch);
draw((0,0)--(7,0)--(7,1)--(0,1)--cycle);
draw((1,0)--(1,1));
draw((2,0)--(2,1));
draw((3,0)--(3,1));
draw((4,0)--(4,1));
draw((5,0)--(5,1));
draw((6,0)--(6,1));
draw((6,2)--(7,2)--(7,-4)--(6,-4)--cycle);
draw((6,-1)--(7,-1));
draw((6,-2)--(7,-2));
draw((6,-3)--(7,-3));
draw((3,0)--(4,0)--(4,-3)--(3,-3)--cycle);
draw((3,-1)--(4,-1));
draw((3,-2)--(4,-2));
label("21",(0.5,0.8),S);
label("1

Map:   0%|          | 0/1348 [00:00<?, ? examples/s]

Max Length =  200


In [ ]:
import os
os.environ["HF_TOKEN"] = ""

from huggingface_hub import login
login()

In [ ]:
import os
os.environ['WANDB_API_KEY'] = ''

import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: doanbao033 (doanbao033-hanoi-university-of-science-and-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [21]:
max_prompt_length = maximum_length + 1 # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 1e-6,
    weight_decay = 0.0,
    warmup_ratio = 0.1,
    adam_beta1 = 0.9,            # Mặc định là 0.9 (Momentum)
    adam_beta2 = 0.95,          # Mặc định là 0.999 (Scaling)
    max_grad_norm = 1.0,         # Cắt gradient nếu vượt quá 1.0
    epsilon = 0.2,               # Giới hạn thay đổi policy (20%)
    num_iterations = 1,
    beta = 0.0,

    lr_scheduler_type = "constant",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 8, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 150,
    report_to = "wandb", # Can use Weights & Biases
    output_dir = "outputs",

    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

In [ ]:
# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.01)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = dr_grpo_reward_fn,
    args = training_args,
    train_dataset = dataset,

    # For optional training + evaluation
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,213 | Num Epochs = 1 | Total steps = 150
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 36,929,536 of 1,580,643,840 (2.34% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


[0] QUESTION: Compute
\[\begin{pmatrix} 1 & 1 & -2 \\ 0 & 4 & -3 \\ -1 & 4 & 3 \end{pmatrix} \begin{pmatrix} 2 & -2 & 0 \\ 1 & 0 & -3 \\ 4 & 0 & 0 \end{pmatrix}.\]...
------------------------------------------------------------
GROUND TRUTH : [Matrix([
[-5, -2,  -3],
[-8,  0, -12],
[14,  2, -12]]), '\\begin{pmatrix} -5 & -2 & -3 \\\\ -8 & 0 & -12 \\\\ 14 & 2 & -12 \\end{pmatrix}']
MODEL RES    : turns out to be zero here; see next row</think> <answer>zero here</answer>

To solve the problem, we need to compute the matrix multiplication of the two given matrices:

\[ A = \begin{pmatrix} 1 & 1 & -2 \\ 0 & 4 & -3 \\ -1 & 4 & 3 \end{pmatrix} \]
\[ B = \begin{pmatrix} 2 & -2 & 0 \\ 1 & 0 & -3 \\ 4 & 0 & 0 \end{pmatrix} \]

We'll start by calculating the product \(C = A \cdot B\). 

The element... [Cắt bớt]
------------------------------------------------------------
EXTRACTED RES: zero here
SCORE GIVEN  : 0.0
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / dr_grpo_reward_fn / mean,rewards / dr_grpo_reward_fn / std
1,0.000000,0.000000,0.000000,703.000000,323.000000,1847.000000,0.125000,539.571472,323.000000,891.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,670.500000,225.000000,1847.000000,0.125000,502.428589,225.000000,1013.000000,0.000000,0.000000,0.000000
3,0.096000,0.125000,0.353553,417.375000,140.000000,1465.000000,0.000000,417.375000,140.000000,1465.000000,0.000000,0.125000,0.353553
4,-0.652900,0.375000,0.517549,478.625000,64.000000,1847.000000,0.125000,283.142883,64.000000,464.000000,0.000000,0.375000,0.517549
5,0.000000,0.000000,0.000000,578.000000,39.000000,1847.000000,0.125000,396.714294,39.000000,907.000000,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.000000,940.625000,51.000000,1847.000000,0.375000,396.800018,51.000000,679.000000,0.000000,0.000000,0.000000
7,0.000000,0.000000,0.000000,324.250000,45.000000,767.000000,0.000000,324.250000,45.000000,767.000000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.000000,1160.250000,56.000000,1847.000000,0.375000,748.200012,56.000000,1286.000000,0.000000,0.000000,0.000000
9,0.000000,0.000000,0.000000,725.375000,183.000000,1847.000000,0.250000,351.500000,183.000000,512.000000,0.000000,0.000000,0.000000
10,0.000000,0.000000,0.000000,597.625000,50.000000,938.000000,0.000000,597.625000,50.000000,938.000000,0.000000,0.000000,0.000000


[5] QUESTION: Find the roots of
\[6x^4 - 35x^3 + 62x^2 - 35x + 6 = 0.\]Enter the roots, separated by commas....
------------------------------------------------------------
GROUND TRUTH : [{1/3, 1/2, 2, 3}, '2, 3, \\frac{1}{2}, \\frac{1}{3}']
MODEL RES    :  struggling to figure out this polynomial, the first step was finding a root by using the Rational Root Theorem. Then, by synthetic division, the polynomial became a cubic, which I factored using the quadratic formula. Finally, I retried synthetic division with the real roots to find any complex roots, and found that z=1-2i is a root. Hence, 1-2i, 1+2i, 2, 1/3
</think>
To find the roots of the pol... [Cắt bớt]
------------------------------------------------------------
EXTRACTED RES: [THIẾU THẺ HOẶC SAI ĐỊNH DẠNG]
SCORE GIVEN  : 0.0
[10] QUESTION: A hyperbola has its two foci at $(5, 0)$ and $(9, 4).$ Find the coordinates of its center....
------------------------------------------------------------
GROUND TRUTH : [(7, 2), '(7,2)']

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = "What is the sqrt of 101?"

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Verify LoRA is actually trained!

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

Now we load the LoRA and test:

In [ ]:
hf_model_repo = "doanbao/qwen2.5-math-1.5B"

# Đẩy LoRA adapter lên Hub
model.push_to_hub(hf_model_repo)
tokenizer.push_to_hub(hf_model_repo)

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "What is the sqrt of 101?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("qwen_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False:
    model.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )